In [ ]:
# Cài thư viện — chạy 1 lần duy nhất trong Terminal:
# pip install pymupdf sentence-transformers chromadb langchain langchain-community google-generativeai python-dotenv google-genai tqdm streamlit


In [1]:
# Kiểm tra các thư viện quan trọng đã cài chưa
import importlib

libs = ["fitz", "sentence_transformers", "chromadb", "langchain", "google.generativeai"]

for lib in libs:
    try:
        importlib.import_module(lib)
        print(f"✅ {lib}")
    except ImportError:
        print(f"❌ {lib} — cần cài lại")

✅ fitz


C:\Users\USER\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ sentence_transformers
✅ chromadb
✅ langchain
✅ google.generativeai


C:\Program Files\WindowsApps\PythonSoftwareFoundation.Python.3.11_3.11.2544.0_x64__qbz5n2kfra8p0\Lib\importlib\__init__.py:126: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  return _bootstrap._gcd_import(name[level:], package, level)


ĐỌC PDF


In [3]:
import fitz  # PyMuPDF
import os

def extract_text_from_pdf(pdf_path):
    """
    Đọc toàn bộ text từ một file PDF.
    FIX: Đóng document ĐÚNG CÁCH
    """
    try:
        doc = fitz.open(pdf_path)
        pages_text = []
        total_pages = len(doc)

        for page_num in range(total_pages):
            try:
                page = doc[page_num]
                text = page.get_text()

                if len(text.strip()) > 50:
                    pages_text.append({
                        "page": page_num + 1,
                        "text": text.strip()
                    })
            except Exception as e:
                print(f"⚠️  Lỗi trang {page_num + 1}: {e}")
                continue

        # QUAN TRỌNG: Đóng document TRƯỚC khi return
        doc.close()

        return {
            "filename": os.path.basename(pdf_path),
            "total_pages": total_pages,
            "pages": pages_text,
            "success": True
        }

    except Exception as e:
        print(f"❌ Lỗi: {e}")
        return {"filename": os.path.basename(pdf_path), "pages": [], "success": False}


# ===== CHẠY =====
PDF_FOLDER = './data/pdfs/'
pdf_files = [f for f in os.listdir(PDF_FOLDER) if f.endswith('.pdf')]

print(f"📚 Tìm thấy {len(pdf_files)} file PDF\n")

all_documents = []
for filename in pdf_files:
    filepath = PDF_FOLDER + filename
    result = extract_text_from_pdf(filepath)
    if result["success"]:
        all_documents.append(result)
        print(f"✅ {filename} — {len(result['pages'])} trang")

print(f"\n✨ Đã đọc xong {len(all_documents)} file!")

📚 Tìm thấy 20 file PDF

✅ 02050003862.pdf — 20 trang
✅ 04 Nguyen Thi Nghia.pdf — 10 trang
✅ 0904ff44-a18b-45be-be69-860c36869afe.pdf — 61 trang
✅ 4.1. Tom tat Luan an_Tieng Viet (24tr).pdf — 27 trang
✅ 42796-Article Text-135454-1-10-20191003.pdf — 8 trang
✅ 48nguyen_thi_thuy_huyen_pham_thanh_tam_nguyen_thi_lien_6798.pdf — 6 trang
✅ 4a-Trich yeu Luan an Marketing-tieng Viet.pdf — 2 trang
✅ article_1741444176.pdf — 9 trang
✅ depression.pdf — 7 trang
✅ DT071-Ứng-dụng-học-máy-phân-tích-dữ-liệu-Chính-phủ-điện-tử-hướng-tới-chuyển-đổi-số-bền-vững.docx.pdf — 52 trang
✅ Enhancing Cyber Resilience_ Development Challenges and Strategi.pdf — 25 trang
✅ jabes-06-2024-0300.pdf — 12 trang
✅ KHNNQS-45-9.2024-11.Nguyen-Thi-Hoan.pdf — 9 trang
✅ pjss-25-752.pdf — 8 trang
✅ research-paper-on-cyber-security-770o245z8oyi.pdf — 8 trang
✅ tesi.pdf — 57 trang
✅ The Impact of Racism on the Personal and Professional Lives of St.pdf — 205 trang
✅ Thực-hiện-pháp-luật-về-dân-chủ-cơ-sở-từ-thực-tiễn-huyện-Cần-Giuộc-t

CHUNKING

In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Cấu hình chunking
# chunk_size=800: mỗi đoạn ~800 ký tự (~150-200 từ tiếng Việt)
# chunk_overlap=100: chồng lấp 100 ký tự giữa các đoạn
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100,
    separators=["\n\n", "\n", ".", " ", ""]
)

def create_chunks(all_documents):
    """
    Tạo chunks từ toàn bộ tài liệu.
    Mỗi chunk lưu kèm metadata: tên file, số trang, chunk ID.
    """
    all_chunks = []
    chunk_id = 0

    for doc in all_documents:
        for page_data in doc["pages"]:
            # Cắt text của trang này thành nhiều chunk
            chunks = text_splitter.split_text(page_data["text"])

            for chunk_text in chunks:
                # Bỏ qua chunk quá ngắn (ít hơn 50 ký tự)
                if len(chunk_text.strip()) < 50:
                    continue

                all_chunks.append({
                    "id": f"chunk_{chunk_id}",
                    "text": chunk_text.strip(),
                    "metadata": {
                        "source": doc["filename"],
                        "page": page_data["page"]
                    }
                })
                chunk_id += 1

    return all_chunks

# Tạo chunks
print("⏳ Đang cắt văn bản thành chunks...")
chunks = create_chunks(all_documents)

# Kiểm tra kết quả
print(f"✅ Đã tạo {len(chunks)} chunks từ {len(all_documents)} luận văn\n")

print(f"📊 Thống kê chunks:")
print(f"   - Chunk ngắn nhất: {min(len(c['text']) for c in chunks)} ký tự")
print(f"   - Chunk dài nhất: {max(len(c['text']) for c in chunks)} ký tự")
print(f"   - Trung bình: {sum(len(c['text']) for c in chunks) // len(chunks)} ký tự\n")

# Preview chunk đầu tiên
print(f"--- Preview Chunk #1 ---")
print(f"ID: {chunks[0]['id']}")
print(f"Nguồn: {chunks[0]['metadata']['source']} - Trang {chunks[0]['metadata']['page']}")
print(f"Độ dài: {len(chunks[0]['text'])} ký tự")
print(f"\nNội dung:")
print(f"{'─'*60}")
print(chunks[0]['text'][:300] + "...")
print(f"{'─'*60}")

print(f"\n✨ Cell 4 hoàn thành! Sẵn sàng chạy Cell 5 (Load Embedding Model)")



⏳ Đang cắt văn bản thành chunks...
✅ Đã tạo 2213 chunks từ 20 luận văn

📊 Thống kê chunks:
   - Chunk ngắn nhất: 63 ký tự
   - Chunk dài nhất: 800 ký tự
   - Trung bình: 669 ký tự

--- Preview Chunk #1 ---
ID: chunk_0
Nguồn: 02050003862.pdf - Trang 1
Độ dài: 260 ký tự

Nội dung:
────────────────────────────────────────────────────────────
ĐẠI HỌC QUỐC GIA HÀ NỘI 
TRƯỜNG ĐẠI HỌC KHOA HỌC XÃ HỘI VÀ NHÂN VĂN 
KHOA TRIẾT HỌC 
---------------------- 
 
 
 
ĐOÀN VĂN NAM 
 
 
 
 
XÂY DỰNG ĐỜI SỐNG VĂN HÓA TINH THẦN  
Ở TỈNH BẮC GIANG HIỆN NAY 
 
 
 
LUẬN VĂN THẠC SĨ 
 
 
 
 
 
 
 
 
 
BẮC GIANG - 2015...
────────────────────────────────────────────────────────────

✨ Cell 4 hoàn thành! Sẵn sàng chạy Cell 5 (Load Embedding Model)


Load Embedding Model

In [5]:
from sentence_transformers import SentenceTransformer

print("⏳ Đang tải embedding model (~120MB, chờ 1-2 phút lần đầu)...")

embedding_model = SentenceTransformer(
    'paraphrase-multilingual-MiniLM-L12-v2'
)

print("✅ Model đã sẵn sàng!")

# Test nhanh
test_vec = embedding_model.encode("Trí tuệ nhân tạo trong giáo dục")
print(f"   Vector dimension: {len(test_vec)} chiều")  # Sẽ ra 384
print(f"   Ví dụ 5 số đầu: {test_vec[:5]}")

print("\n✨ Sẵn sàng cho Cell 6 (Lưu vào ChromaDB)!")

⏳ Đang tải embedding model (~120MB, chờ 1-2 phút lần đầu)...


C:\Users\USER\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\USER\.cache\huggingface\hub\models--sentence-transformers--paraphrase-multilingual-MiniLM-L12-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn

✅ Model đã sẵn sàng!
   Vector dimension: 384 chiều
   Ví dụ 5 số đầu: [-0.08982962 -0.16844302 -0.15661716 -0.23248397 -0.06829064]

✨ Sẵn sàng cho Cell 6 (Lưu vào ChromaDB)!


embed và lưu vào ChromaDB

In [6]:
import chromadb
from tqdm import tqdm

# Kết nối ChromaDB, lưu trên Drive
DB_PATH = './chroma_db'
client = chromadb.PersistentClient(path=DB_PATH)

# Tạo collection
collection = client.get_or_create_collection(
    name="thesis_knowledge",
    metadata={"hnsw:space": "cosine"}
)

print(f"📦 Collection hiện có: {collection.count()} chunks")
print(f"⏳ Bắt đầu embed và lưu {len(chunks)} chunks...\n")

# Xử lý theo batch để tránh out-of-memory
BATCH_SIZE = 50
total_batches = (len(chunks) + BATCH_SIZE - 1) // BATCH_SIZE

for i in tqdm(range(0, len(chunks), BATCH_SIZE),
              desc="Embedding", total=total_batches):
    batch = chunks[i:i + BATCH_SIZE]

    ids        = [c["id"] for c in batch]
    texts      = [c["text"] for c in batch]
    metadatas  = [c["metadata"] for c in batch]

    # Chuyển text → vector
    embeddings = embedding_model.encode(texts).tolist()

    # Lưu vào ChromaDB
    collection.add(
        ids=ids,
        documents=texts,
        embeddings=embeddings,
        metadatas=metadatas
    )

print(f"\n✅ Hoàn thành! Database có {collection.count()} chunks")
print(f"💾 Đã lưu vào: {DB_PATH}")
print(f"\n✨ Sẵn sàng cho Cell 7 (Test Query)!")

📦 Collection hiện có: 0 chunks
⏳ Bắt đầu embed và lưu 2213 chunks...



Embedding: 100%|██████████| 45/45 [00:54<00:00,  1.20s/it]


✅ Hoàn thành! Database có 2213 chunks
💾 Đã lưu vào: ./chroma_db

✨ Sẵn sàng cho Cell 7 (Test Query)!


Cell 7: Test Querry


In [7]:
# ===== TEST QUERY =====
print("🔍 Bắt đầu test query ChromaDB...\n")

# Danh sách câu hỏi test (cả tiếng Việt và tiếng Anh)
test_questions = [
    "trí tuệ nhân tạo ứng dụng trong giáo dục",
    "phương pháp nghiên cứu định tính",
    "machine learning deep learning",
]

for question in test_questions:
    print(f"{'='*60}")
    print(f"❓ Câu hỏi: {question}")
    print(f"{'='*60}")

    # Embed câu hỏi
    question_vector = embedding_model.encode(question).tolist()

    # Tìm 3 đoạn liên quan nhất
    results = collection.query(
        query_embeddings=[question_vector],
        n_results=3
    )

    for i, (doc, meta) in enumerate(zip(
        results['documents'][0],
        results['metadatas'][0]
    )):
        print(f"\n📄 Kết quả {i+1}:")
        print(f"   Nguồn: {meta['source']} - Trang {meta['page']}")
        print(f"   Nội dung: {doc[:200]}...")

    print()

print("✅ Test hoàn thành!")
print("🎉 GIAI ĐOẠN 1 HOÀN TẤT — Sẵn sàng sang Giai đoạn 2 (Kết nối Gemini)!")

🔍 Bắt đầu test query ChromaDB...

❓ Câu hỏi: trí tuệ nhân tạo ứng dụng trong giáo dục

📄 Kết quả 1:
   Nguồn: jabes-06-2024-0300.pdf - Trang 9
   Nội dung: future. Students can leverage this knowledge to examine areas like machine learning, deep
learning and natural language processing, gaining a competitive edge in professional tasks.
Universities can f...

📄 Kết quả 2:
   Nguồn: jabes-06-2024-0300.pdf - Trang 1
   Nội dung: subjects to improve students’ AI proficiency and capacity.
Originality/value – This study examines the determinants of AI adoption by accounting students in Vietnam,
addressing a previously unexplored...

📄 Kết quả 3:
   Nguồn: jabes-06-2024-0300.pdf - Trang 3
   Nội dung: individual students, offering personalised learning approaches. AI-powered libraries improve
the educational experience by providing better access to resources in higher education
institutions (Cox et...

❓ Câu hỏi: phương pháp nghiên cứu định tính

📄 Kết quả 1:
   Nguồn: 4.1. Tom tat Luan an_Ti

 Lấy API key Gemini & kết nối

In [8]:
# CELL 8: Khởi tạo đầu bếp Gemini (Phiên bản mới nhất)

from google import genai
import getpass

# 1. Nhập API Key bảo mật
GEMINI_API_KEY = getpass.getpass("🔑 Nhập Gemini API Key của bạn: ")

# 2. Khởi tạo Client theo chuẩn thư viện mới
client = genai.Client(api_key=GEMINI_API_KEY)

# 3. Test kết nối với mô hình đời mới (gemini-2.5-flash hoặc gemini-3-flash-preview)
print("⏳ Đang test kết nối Gemini...")
try:
    test_response = client.models.generate_content(
        model='gemini-2.5-flash',
        contents="Xin chào! Trả lời bằng 1 câu ngắn."
    )
    print(f"✅ Gemini phản hồi: {test_response.text}")
    print("\n✨ Kết nối thành công! 'Đầu bếp' đã sẵn sàng cho Giai đoạn 2.")
except Exception as e:
    print(f"❌ Lỗi kết nối: {e}")
    print("Vui lòng kiểm tra lại API Key hoặc tên Mô hình.")

⏳ Đang test kết nối Gemini...
✅ Gemini phản hồi: Chào bạn!

✨ Kết nối thành công! 'Đầu bếp' đã sẵn sàng cho Giai đoạn 2.


Cell 9: RAG


In [9]:
# CELL 9: RAG Pipeline — Kết nối ChromaDB + Gemini

from google import genai

def rag_query(question, n_results=3):
    """
    RAG Pipeline hoàn chỉnh:
    1. Embed câu hỏi → vector
    2. Tìm chunks liên quan từ ChromaDB
    3. Gửi chunks + câu hỏi cho Gemini
    4. Trả về câu trả lời có trích dẫn nguồn
    """

    # ── BƯỚC 1: Embed câu hỏi ──────────────────────────
    question_vector = embedding_model.encode(question).tolist()

    # ── BƯỚC 2: Tìm chunks liên quan từ ChromaDB ───────
    results = collection.query(
        query_embeddings=[question_vector],
        n_results=n_results
    )

    chunks_text = results['documents'][0]
    chunks_meta = results['metadatas'][0]

    # ── BƯỚC 3: Tạo context từ chunks ──────────────────
    context = ""
    sources = []

    for i, (chunk, meta) in enumerate(zip(chunks_text, chunks_meta)):
        context += f"\n[Đoạn {i+1}] Nguồn: {meta['source']} - Trang {meta['page']}\n"
        context += f"{chunk}\n"
        context += "─" * 40 + "\n"
        sources.append(f"{meta['source']} (Trang {meta['page']})")

    # ── BƯỚC 4: Tạo prompt gửi Gemini ──────────────────
    prompt = f"""Bạn là trợ lý nghiên cứu học thuật thông minh,
chuyên hỗ trợ sinh viên tìm kiếm và kế thừa tri thức từ luận văn.

Dựa trên các đoạn trích từ luận văn sau đây:
{context}

Hãy trả lời câu hỏi: {question}

Yêu cầu:
- Chỉ sử dụng thông tin từ các đoạn trích trên
- Trả lời rõ ràng, súc tích bằng tiếng Việt
- Cuối câu trả lời, ghi rõ nguồn trích dẫn
- Nếu thông tin không đủ, hãy nói rõ điều đó
"""

    # ── BƯỚC 5: Gọi Gemini sinh câu trả lời ───────────
    response = client.models.generate_content(
        model='gemini-2.5-flash',
        contents=prompt
    )

    return {
        "question": question,
        "answer": response.text,
        "sources": list(set(sources)),
        "chunks_used": len(chunks_text)
    }


# ── TEST THỬ ────────────────────────────────────────────
print("🧪 Test RAG Pipeline...\n")
print("=" * 60)

test_q = "Nói thêm luật pháp dân chủ cơ sở ở Long An"
result = rag_query(test_q)

print(f"❓ Câu hỏi: {result['question']}")
print(f"\n💬 Câu trả lời:\n{result['answer']}")
print(f"\n📚 Nguồn tham khảo ({result['chunks_used']} đoạn):")
for src in result['sources']:
    print(f"   - {src}")

print("\n✅ RAG Pipeline hoạt động!")
print("✨ Sẵn sàng cho Cell 10 (Chatbot Loop)!")

🧪 Test RAG Pipeline...

❓ Câu hỏi: Nói thêm luật pháp dân chủ cơ sở ở Long An

💬 Câu trả lời:
Dựa trên các đoạn trích, luật pháp dân chủ cơ sở ở tỉnh Long An (cụ thể là huyện Cần Giuộc) được thực hiện chủ yếu dựa trên các quy định của **Pháp lệnh số 34/2007/PL-UBTVQH11 ngày 20/4/2007**, gồm 6 Chương với 28 điều. Đây là các hoạt động nhằm đưa quy định của Pháp lệnh này vào cuộc sống trên địa bàn.

Bên cạnh đó, việc thực hiện còn chịu ảnh hưởng và được hướng dẫn bởi các văn bản chỉ đạo khác như:
*   Chỉ thị số 30-CT/TW của Bộ Chính trị (khóa VIII) ngày 18/02/1998 về xây dựng và thực hiện Quy chế dân chủ (QCDC) ở cơ sở, cùng với các văn bản tiếp tục thực hiện như Chỉ thị số 10-CT/TW ngày 28/3/2002 của Ban Bí thư Trung ương Đảng (khóa IX) và Kết luận số 65-KL/TW của Ban Bí thư (khóa X).
*   Các Nghị định 29, 79 của Chính phủ.
*   Nghị quyết số 55/NQ-UBTVQH10 ngày 30/7/1998.
*   Trên cơ sở các văn bản chỉ đạo quốc gia, Tỉnh ủy và UBND tỉnh Long An cũng ban hành các văn bản chỉ đạo riêng để 

CELL 10: Chatbot loop

In [11]:
# CELL 10: Chatbot Loop — FINAL VERSION
# Fix: Context window 30 lượt + Safety Check + Indentation đúng

import time
import re
from google.genai import types


def rag_query_with_context(question, chat_history, n_results=5):
    """
    RAG Pipeline với Context Window 30 lượt + Safety Check hallucination
    """

    # ── BƯỚC 1: Embed câu hỏi ──────────────────────────
    question_vector = embedding_model.encode(question).tolist()

    # ── BƯỚC 2: Tìm chunks liên quan ───────────────────
    results = collection.query(
        query_embeddings=[question_vector],
        n_results=n_results
    )

    chunks_text = results['documents'][0]
    chunks_meta = results['metadatas'][0]

    # ── BƯỚC 3: Tạo knowledge context ──────────────────
    knowledge_context = ""
    sources = []

    for i, (chunk, meta) in enumerate(zip(chunks_text, chunks_meta)):
        knowledge_context += f"\n[Đoạn {i+1}] Nguồn: {meta['source']} - Trang {meta['page']}\n"
        knowledge_context += f"{chunk}\n"
        knowledge_context += "─" * 40 + "\n"
        sources.append(f"{meta['source']} (Trang {meta['page']})")

    # ── BƯỚC 4: Tạo conversation context ───────────────
    conversation_context = ""
    if chat_history:
        recent = chat_history[-30:]
        conversation_context = "\n=== LỊCH SỬ HỘI THOẠI ===\n"
        for i, h in enumerate(recent, 1):
            conversation_context += f"\nLượt {i}:\n"
            conversation_context += f"Người dùng: {h['question']}\n"
            conversation_context += f"Trợ lý: {h['answer']}\n"
        conversation_context += "=== KẾT THÚC LỊCH SỬ ===\n"

    # ── BƯỚC 5: Prompt ─────────────────────────────────
    prompt = f"""Bạn là trợ lý nghiên cứu học thuật của UEH.
Nhiệm vụ DUY NHẤT của bạn là tổng hợp thông tin từ các đoạn trích được cung cấp.

{conversation_context}

=== TRI THỨC TỪ KHO LUẬN VĂN ===
{knowledge_context}
=== KẾT THÚC TRI THỨC ===

Câu hỏi: {question}

LUẬT BẮT BUỘC — VI PHẠM LÀ KHÔNG ĐƯỢC CHẤP NHẬN:
1. CHỈ dùng thông tin có trong các đoạn trích trên
2. TUYỆT ĐỐI KHÔNG thêm citations, references, tên tác giả ngoài những gì xuất hiện trong đoạn trích
3. TUYỆT ĐỐI KHÔNG dùng kiến thức nền từ training data
4. Nếu thông tin không có trong đoạn trích → nói rõ "Kho dữ liệu hiện tại chưa có thông tin về vấn đề này"
5. Mọi claim phải có thể truy về đoạn trích cụ thể ([Đoạn 1], [Đoạn 2]...)
6. KHÔNG được suy diễn hoặc mở rộng ra ngoài nội dung đoạn trích

FORMAT TRẢ LỜI:
- Trả lời đầy đủ, có cấu trúc rõ ràng
- Ghi rõ [Đoạn X] sau mỗi thông tin trích dẫn
- Cuối bài chỉ liệt kê nguồn đã xuất hiện trong đoạn trích, không thêm gì khác
- Trả lời tiếng Việt trừ khi user hỏi tiếng Anh
"""

    # ── BƯỚC 6: Gọi Gemini ─────────────────────────────
    max_retries = 3
    for attempt in range(max_retries):
        try:
            response = client.models.generate_content(
                model='gemini-2.5-flash',
                contents=prompt,
                config=types.GenerateContentConfig(
                    max_output_tokens=8192,
                    temperature=0.3,
                )
            )

            # ── BƯỚC 7: SAFETY CHECK ───────────────────
            raw_answer = response.text

            # Detect citations dạng APA: "Author (2024)" hoặc "Author, A. (2024)"
            citation_pattern = r'[A-Z][a-z]+(?:,\s[A-Z]\.)?\s*\(\d{4}\)'
            citations_in_answer = re.findall(citation_pattern, raw_answer)

            # So sánh với chunks thực tế
            hallucinated = []
            for citation in citations_in_answer:
                found_in_chunks = any(citation in chunk for chunk in chunks_text)
                if not found_in_chunks:
                    hallucinated.append(citation)

            # Gắn warning nếu có hallucination
            if hallucinated:
                warning = (
                    f"\n\n---\n"
                    f"⚠️ **CẢNH BÁO:** Phát hiện {len(hallucinated)} citation "
                    f"có thể KHÔNG có trong tài liệu gốc:\n"
                    + "\n".join([f"  - {c}" for c in hallucinated])
                    + "\n\n_Vui lòng kiểm tra lại trước khi dùng cho báo cáo học thuật._"
                )
                final_answer = raw_answer + warning
                print(f"⚠️  Safety Check: {len(hallucinated)} hallucinated citations detected!")
            else:
                final_answer = raw_answer
                print(f"✅ Safety Check: OK — không phát hiện citations ngoài tài liệu")
            # ── KẾT THÚC SAFETY CHECK ──────────────────

            return {
                "question": question,
                "answer": final_answer,
                "sources": list(set(sources)),
                "chunks_used": len(chunks_text),
                "hallucinated_citations": hallucinated
            }

        except Exception as e:
            if "503" in str(e) or "UNAVAILABLE" in str(e):
                wait_time = (attempt + 1) * 10
                print(f"⚠️  Server busy, thử lại sau {wait_time}s... ({attempt+1}/{max_retries})")
                time.sleep(wait_time)
            else:
                raise e

    return {"error": "Server không phản hồi sau 3 lần thử"}


# ════════════════════════════════════════════════════════
# CHATBOT LOOP
# ════════════════════════════════════════════════════════

def chat_loop():

    print("=" * 60)
    print("🎓 THESIS KNOWLEDGE CHATBOT — UEH")
    print("   (Final Version — Context 30 lượt)")
    print("=" * 60)
    print("💡 Chatbot nhớ tối đa 30 câu hỏi gần nhất")
    print("💡 Gõ 'exit' hoặc 'thoát' để kết thúc")
    print("💡 Gõ 'history' để xem lịch sử")
    print("💡 Gõ 'clear' để xóa lịch sử, bắt đầu mới")
    print("=" * 60 + "\n")

    chat_history = []
    question_count = 0

    while True:
        try:
            user_input = input("👤 Bạn: ").strip()
        except EOFError:
            break

        if not user_input:
            print("💬 Vui lòng nhập câu hỏi!\n")
            continue

        # ── Lệnh đặc biệt ───────────────────────────────
        if user_input.lower() in ['exit', 'thoát', 'quit']:
            print(f"\n👋 Kết thúc phiên chat!")
            print(f"📊 Tổng câu hỏi: {question_count}")
            break

        if user_input.lower() == 'history':
            if not chat_history:
                print("📭 Chưa có lịch sử chat\n")
            else:
                print(f"\n📜 LỊCH SỬ ({len(chat_history)} câu hỏi):")
                print("─" * 40)
                for i, h in enumerate(chat_history, 1):
                    print(f"{i}. ❓ {h['question']}")
                    print(f"   ⏱️  {h['time']}s | 📚 {len(h['sources'])} nguồn\n")
            continue

        if user_input.lower() == 'clear':
            chat_history = []
            question_count = 0
            print("🧹 Đã xóa lịch sử! Bắt đầu cuộc trò chuyện mới.\n")
            continue

        # ── Xử lý câu hỏi ───────────────────────────────
        question_count += 1

        if chat_history:
            print(f"🧠 Context: đang nhớ {min(len(chat_history), 30)} lượt hội thoại...")

        print(f"🔍 Đang tìm kiếm trong kho luận văn ({min(len(chat_history), 30) * '█' or '░'})\n")

        start_time = time.time()

        try:
            result = rag_query_with_context(
                question=user_input,
                chat_history=chat_history
            )
            elapsed = round(time.time() - start_time, 2)

            if "error" in result:
                print(f"❌ {result['error']}\n")
                continue

            print("─" * 60)
            print(f"🤖 Chatbot:\n")
            print(result['answer'])
            print("─" * 60)

            print(f"\n📚 Nguồn tham khảo ({result['chunks_used']} đoạn):")
            for src in result['sources']:
                print(f"   📄 {src}")

            print(f"\n⏱️  Thời gian phản hồi: {elapsed}s")
            print(f"🧠 Context window: {min(len(chat_history) + 1, 30)}/30 lượt\n")

            chat_history.append({
                "question": user_input,
                "answer": result['answer'],
                "sources": result['sources'],
                "time": elapsed
            })

        except Exception as e:
            print(f"❌ Lỗi: {e}")
            print("💡 Thử hỏi lại sau vài giây\n")


# ── CHẠY CHATBOT ──────────────────────────────────────────
chat_loop()

🎓 THESIS KNOWLEDGE CHATBOT — UEH
   (Final Version — Context 30 lượt)
💡 Chatbot nhớ tối đa 30 câu hỏi gần nhất
💡 Gõ 'exit' hoặc 'thoát' để kết thúc
💡 Gõ 'history' để xem lịch sử
💡 Gõ 'clear' để xóa lịch sử, bắt đầu mới

💬 Vui lòng nhập câu hỏi!

💬 Vui lòng nhập câu hỏi!

🔍 Đang tìm kiếm trong kho luận văn (░)

✅ Safety Check: OK — không phát hiện citations ngoài tài liệu
────────────────────────────────────────────────────────────
🤖 Chatbot:

Dưới đây là tổng hợp thông tin từ các đoạn trích được cung cấp:

**1. Phương pháp nghiên cứu tường thuật (Narrative Inquiry)**
Phương pháp nghiên cứu tường thuật được mô tả là một cách tư duy về trải nghiệm, trong đó câu chuyện là một cổng để một người bước vào thế giới và qua đó trải nghiệm thế giới được diễn giải và trở nên có ý nghĩa cá nhân [Đoạn 1]. Phương pháp này cho phép nhà nghiên cứu đồng hành cùng câu chuyện của những người tham gia và cùng xây dựng một bức tranh tổng thể về cách phân biệt chủng tộc trong khuôn viên trường được trải ngh

UI bằng Streamlit


In [12]:
# streamlit đã được cài trong bước 1 — bỏ qua cell này


cell 12: tạo file app.py

In [13]:
# app.py đã được tạo sẵn trong thư mục dự án.
# Mở file app.py riêng để xem/chỉnh sửa.
print('✅ app.py đã sẵn sàng — chạy lệnh: streamlit run app.py')


✅ app.py đã sẵn sàng — chạy lệnh: streamlit run app.py


Cell 13: Set API và chạy

In [14]:
# CELL 13 — Chạy giao diện Streamlit
# Thay vì dùng ngrok, chạy lệnh sau trong Terminal của VS Code:
#
#   streamlit run app.py
#
# Trình duyệt sẽ tự mở tại: http://localhost:8501

import os
import getpass

os.environ["GEMINI_API_KEY"] = getpass.getpass("🔑 Nhập Gemini API Key của bạn: ")
print("✅ API key đã được set! Mở Terminal và chạy: streamlit run app.py")


✅ API key đã được set! Mở Terminal và chạy: streamlit run app.py
